# AWS LSTM Autoencoder — Kaggle train / val

Trains a **bottleneck multivariate LSTM autoencoder** on clean hourly T / RH / P.

| Split | Period | This notebook |
|---|---|---|
| Train | 2020-01-01 → 2022-12-31 | used |
| Val | 2023-01-01 → 2023-12-31 | used (early stop + error distribution) |
| Test | 2024-01-01 → 2024-12-31 | **not loaded, not touched** |

Not in this notebook: simulator, fault injection, Precision/Recall, buddy check, FastAPI.

Download `/kaggle/working/artifacts/` after the run and put it in local `ml/artifacts/` for `testing_model.py`.


In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
from pathlib import Path

SEED = 42

FEATURES = ["temp", "rhum", "pres"]
WINDOW_HOURS = 24

TRAIN_START = "2020-01-01"
TRAIN_END = "2022-12-31 23:00:00"
VAL_START = "2023-01-01"
VAL_END = "2023-12-31 23:00:00"
# Test year is intentionally unused.
TEST_START = "2024-01-01"
TEST_END = "2024-12-31 23:00:00"

INTERPOLATE_LIMIT_HOURS = 2

# Stride 3 keeps ~1/3 of overlapping windows. Set to 1 for denser training.
TRAIN_STRIDE = 3
VAL_MONITOR_MAX_WINDOWS = 65_536
VAL_FINAL_STRIDE = 3

HIDDEN = 64
LATENT = 32
BATCH_SIZE = 256
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 7
GRAD_CLIP = 1.0
NUM_WORKERS = 2 if Path("/kaggle/working").exists() else 0

RAW_CANDIDATES = [
    Path("/kaggle/input/datasets/aviksuom/automatic-weather-stations-data/data/raw"),
    Path("/kaggle/input/automatic-weather-stations-data/data/raw"),
    Path("/kaggle/input/automatic-weather-stations-data"),
    Path("data/raw"),
    Path("/kaggle/working/data/raw"),
]

OUT_DIR = Path("/kaggle/working/artifacts")
if not Path("/kaggle/working").exists():
    OUT_DIR = Path("ml/artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR.resolve())


In [ ]:
# =============================================================================
# IMPORTS / SEEDS / DEVICE
# =============================================================================
import json
import math
import os
import random
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICE = torch.device("cuda:0" if N_GPUS else "cpu")
print("device:", DEVICE, "| gpu_count:", N_GPUS)
if N_GPUS:
    for i in range(N_GPUS):
        print(f"  gpu {i}:", torch.cuda.get_device_name(i))


In [ ]:
# =============================================================================
# RESOLVE DATA PATH
# =============================================================================
def resolve_raw_dir():
    for p in RAW_CANDIDATES:
        if (p / "stations.csv").exists():
            return p
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        hits = list(kaggle_input.rglob("stations.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError(
        "Could not find stations.csv. Add the dataset and check RAW_CANDIDATES."
    )


RAW_DIR = resolve_raw_dir()
print("RAW_DIR:", RAW_DIR)

catalog = pd.read_csv(RAW_DIR / "stations.csv", dtype={"station_id": str})
catalog["station_id"] = catalog["station_id"].astype(str)
STATION_IDS = catalog["station_id"].tolist()
print("stations:", len(STATION_IDS))
catalog.head()


## Load hourly series

Each station file is already hourly-reindexed for 2020–2024. Empty cells are true missing hours.


In [ ]:
def load_station_csv(station_id: str) -> pd.DataFrame:
    path = RAW_DIR / f"{station_id}.csv"
    df = pd.read_csv(path, parse_dates=["timestamp"])
    if "station_id" not in df.columns:
        df["station_id"] = station_id
    df["station_id"] = df["station_id"].astype(str)
    df = df.sort_values("timestamp")
    df = df.drop_duplicates(subset=["timestamp"], keep="first")
    df = df.set_index("timestamp")
    return df[["station_id", *FEATURES]]


raw_by_station = {}
load_errors = []
for sid in tqdm(STATION_IDS, desc="load stations"):
    try:
        raw_by_station[sid] = load_station_csv(sid)
    except Exception as exc:
        load_errors.append((sid, str(exc)))

print("loaded:", len(raw_by_station), "| failures:", len(load_errors))
if load_errors:
    print(load_errors[:10])


## Quality audit + short-gap interpolation

- Linear interpolate **at most 2 consecutive missing hours** (inside the series only).
- Longer holes stay missing and **break** 24-hour windows.
- Windows never cross the train/val calendar boundary.


In [ ]:
def interpolate_short_gaps(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in FEATURES:
        out[col] = out[col].interpolate(
            method="linear",
            limit=INTERPOLATE_LIMIT_HOURS,
            limit_area="inside",
        )
    return out


def complete_mask(df: pd.DataFrame) -> pd.Series:
    return df[FEATURES].notna().all(axis=1)


def slice_period(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    return df.loc[start:end].copy()


# ---- diagnostics on raw (before interpolate) ----
diag_rows = []
for sid, df in raw_by_station.items():
    n = len(df)
    complete = int(complete_mask(df).sum())
    diag_rows.append(
        {
            "station_id": sid,
            "rows": n,
            "complete_raw": complete,
            "complete_raw_pct": round(100.0 * complete / n, 3) if n else 0.0,
            "temp_nan": int(df["temp"].isna().sum()),
            "rhum_nan": int(df["rhum"].isna().sum()),
            "pres_nan": int(df["pres"].isna().sum()),
        }
    )
diag = pd.DataFrame(diag_rows).merge(
    catalog[["station_id", "station_name", "complete_pct"]],
    on="station_id",
    how="left",
)
print(diag[["complete_raw_pct"]].describe().T)
print("stations with 0 complete raw hours:", int((diag["complete_raw"] == 0).sum()))
diag.sort_values("complete_raw_pct").head(8)


In [ ]:
prepared = {}
for sid, df in tqdm(raw_by_station.items(), desc="interpolate"):
    prepared[sid] = interpolate_short_gaps(df)

# freeze-run diagnostic (review only — not used as a drop rule)
freeze_rows = []
for sid, df in prepared.items():
    for col in FEATURES:
        s = df[col]
        valid = s.notna()
        if valid.sum() < 24:
            continue
        same = s.eq(s.shift()) & valid & valid.shift(fill_value=False)
        # run lengths of consecutive equal values
        grp = (~same).cumsum()
        run = same.groupby(grp).transform("size")
        max_run = int(run.max()) if len(run) else 0
        freeze_rows.append({"station_id": sid, "feature": col, "max_equal_run_hours": max_run})

freeze = pd.DataFrame(freeze_rows)
print("longest equal-value runs (possible freeze or calm weather):")
print(
    freeze.sort_values("max_equal_run_hours", ascending=False)
    .head(12)
    .to_string(index=False)
)


## Chronological split (no shuffle, no 2024)


In [ ]:
train_frames = {}
val_frames = {}
skipped_train = []
skipped_val = []

for sid, df in prepared.items():
    tr = slice_period(df, TRAIN_START, TRAIN_END)
    va = slice_period(df, VAL_START, VAL_END)
    if tr.empty or not complete_mask(tr).any():
        skipped_train.append(sid)
    else:
        train_frames[sid] = tr
    if va.empty or not complete_mask(va).any():
        skipped_val.append(sid)
    else:
        val_frames[sid] = va

print("train stations:", len(train_frames), "| skipped:", skipped_train)
print("val stations  :", len(val_frames), "| skipped:", skipped_val)
print("2024 loaded?  :", False)


## Station-wise MinMax — fit on **train only**


In [ ]:
def fit_minmax(train_df: pd.DataFrame) -> dict:
    # min/max per feature. Constant channels get range 1 to avoid div/0.
    stats = {"features": FEATURES, "min": [], "max": [], "range": []}
    complete = train_df.loc[complete_mask(train_df), FEATURES]
    if complete.empty:
        raise ValueError("no complete train rows")
    for col in FEATURES:
        vmin = float(complete[col].min())
        vmax = float(complete[col].max())
        vr = vmax - vmin
        if (not math.isfinite(vr)) or vr == 0.0:
            vr = 1.0
        stats["min"].append(vmin)
        stats["max"].append(vmax)
        stats["range"].append(vr)
    return stats


def apply_minmax(df: pd.DataFrame, stats: dict) -> np.ndarray:
    arr = df[FEATURES].to_numpy(dtype=np.float64)
    vmin = np.array(stats["min"], dtype=np.float64)
    vr = np.array(stats["range"], dtype=np.float64)
    scaled = (arr - vmin) / vr
    return scaled.astype(np.float32)


scalers = {}
failed_scalers = []
for sid, tr in train_frames.items():
    try:
        scalers[sid] = fit_minmax(tr)
    except Exception as exc:
        failed_scalers.append((sid, str(exc)))

print("scalers:", len(scalers), "| failed:", failed_scalers[:5])

# preview one station
sid0 = next(iter(scalers))
print("example", sid0, scalers[sid0])


In [ ]:
def valid_window_starts(complete: np.ndarray, stride: int, window: int = WINDOW_HOURS):
    # Starts where `window` consecutive hours are all complete.
    n = len(complete)
    if n < window:
        return np.array([], dtype=np.int32)
    c = complete.astype(np.int32)
    cs = np.concatenate([[0], np.cumsum(c)])
    starts = []
    for i in range(0, n - window + 1, stride):
        if cs[i + window] - cs[i] == window:
            starts.append(i)
    return np.asarray(starts, dtype=np.int32)


def build_index(frames: dict, stride: int):
    # List of (station_id, local_row_start). Scaled arrays stored separately.
    scaled = {}
    complete = {}
    index = []
    per_station = {}
    for sid, df in frames.items():
        if sid not in scalers:
            continue
        scaled[sid] = apply_minmax(df, scalers[sid])
        comp = complete_mask(df).to_numpy()
        complete[sid] = comp
        starts = valid_window_starts(comp, stride=stride)
        per_station[sid] = int(len(starts))
        for s in starts:
            index.append((sid, int(s)))
    return scaled, complete, index, per_station


train_scaled, train_complete, train_index, train_counts = build_index(
    train_frames, TRAIN_STRIDE
)
val_scaled, val_complete, val_index, val_counts = build_index(
    val_frames, VAL_FINAL_STRIDE
)

print("train windows:", len(train_index))
print("val windows  :", len(val_index))
print(
    "train windows/station: min",
    min(train_counts.values()) if train_counts else 0,
    "median",
    int(np.median(list(train_counts.values()))) if train_counts else 0,
    "max",
    max(train_counts.values()) if train_counts else 0,
)

# fixed val subset for epoch-level early stopping (not for final percentiles)
rng = np.random.RandomState(SEED)
if len(val_index) > VAL_MONITOR_MAX_WINDOWS:
    pick = rng.choice(len(val_index), size=VAL_MONITOR_MAX_WINDOWS, replace=False)
    val_monitor_index = [val_index[i] for i in pick]
else:
    val_monitor_index = list(val_index)
print("val monitor windows:", len(val_monitor_index))


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, scaled: dict, index: list, window: int = WINDOW_HOURS):
        self.scaled = scaled
        self.index = index
        self.window = window

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        sid, start = self.index[i]
        x = self.scaled[sid][start : start + self.window]
        # remaining NaN should not happen if window filter is correct
        return torch.from_numpy(np.nan_to_num(x, nan=0.0))


train_ds = WindowDataset(train_scaled, train_index)
val_mon_ds = WindowDataset(val_scaled, val_monitor_index)
val_full_ds = WindowDataset(val_scaled, val_index)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=bool(N_GPUS),
    drop_last=True,
)
val_mon_loader = DataLoader(
    val_mon_ds,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=bool(N_GPUS),
)
val_full_loader = DataLoader(
    val_full_ds,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=bool(N_GPUS),
)

xb = next(iter(train_loader))
print("batch shape:", tuple(xb.shape), xb.dtype)
assert xb.shape[1:] == (WINDOW_HOURS, len(FEATURES))


## Model — bottleneck LSTM autoencoder

Encoder last hidden state is compressed to a **32-d latent vector** (not a 24-step latent sequence). Decoder reconstructs the full 24 × 3 window from that vector.


In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features=3, hidden=64, latent=32):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
        )
        self.to_latent = nn.Linear(hidden, latent)
        self.decoder = nn.LSTM(
            input_size=latent,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
        )
        self.to_output = nn.Linear(hidden, n_features)
        self.latent = latent

    def encode(self, x):
        # x: (B, T, F)
        _, (h_n, _) = self.encoder(x)
        z = torch.tanh(self.to_latent(h_n[-1]))  # (B, latent)
        return z

    def decode(self, z, seq_len):
        dec_in = z.unsqueeze(1).repeat(1, seq_len, 1)  # (B, T, latent)
        dec_out, _ = self.decoder(dec_in)
        return self.to_output(dec_out)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z, x.size(1))


def unwrap(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


core = LSTMAutoencoder(
    n_features=len(FEATURES),
    hidden=HIDDEN,
    latent=LATENT,
)
n_params = sum(p.numel() for p in core.parameters())
print("parameters:", n_params)

if N_GPUS > 1:
    model = nn.DataParallel(core)
    print("using DataParallel on", N_GPUS, "GPUs")
else:
    model = core
model = model.to(DEVICE)

with torch.no_grad():
    yb = model(xb.to(DEVICE))
print("output shape:", tuple(yb.shape))
assert yb.shape == xb.to(DEVICE).shape


## Tiny overfit check

If loss does not collapse on 128 windows, stop and fix the pipeline before the full run.


In [ ]:
tiny_idx = train_index[:128]
tiny_loader = DataLoader(WindowDataset(train_scaled, tiny_idx), batch_size=32, shuffle=True)
probe = LSTMAutoencoder(len(FEATURES), HIDDEN, LATENT).to(DEVICE)
opt_p = torch.optim.Adam(probe.parameters(), lr=LR)
crit = nn.MSELoss()
probe.train()
tiny_losses = []
for step in range(80):
    total = 0.0
    n = 0
    for batch in tiny_loader:
        batch = batch.to(DEVICE)
        opt_p.zero_grad(set_to_none=True)
        recon = probe(batch)
        loss = crit(recon, batch)
        loss.backward()
        opt_p.step()
        total += loss.item() * batch.size(0)
        n += batch.size(0)
    tiny_losses.append(total / max(n, 1))

print(f"tiny loss start={tiny_losses[0]:.6f}  end={tiny_losses[-1]:.6f}")
if tiny_losses[-1] > 0.05 and tiny_losses[-1] > 0.5 * tiny_losses[0]:
    print("WARNING: tiny set did not overfit. Inspect scaling / shapes before full training.")
else:
    print("tiny overfit looks OK")

plt.figure(figsize=(6, 3))
plt.plot(tiny_losses)
plt.title("Tiny-set overfit MSE")
plt.xlabel("step")
plt.ylabel("MSE")
plt.tight_layout()
plt.show()

del probe, opt_p
torch.cuda.empty_cache() if N_GPUS else None


## Full training

Early stopping on a **fixed val subset**. Final val error distribution is computed after training on the full val index.


In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    unwrap(model).parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-5
)


@torch.no_grad()
def eval_loss(loader) -> float:
    model.eval()
    total = 0.0
    n = 0
    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        recon = model(batch)
        loss = criterion(recon, batch)
        total += loss.item() * batch.size(0)
        n += batch.size(0)
    return total / max(n, 1)


history = []
best_val = float("inf")
best_state = None
best_epoch = -1
stale = 0
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    seen = 0
    pbar = tqdm(train_loader, desc=f"epoch {epoch:02d}/{EPOCHS}", leave=False)
    for batch in pbar:
        batch = batch.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        running += loss.item() * batch.size(0)
        seen += batch.size(0)
        pbar.set_postfix(loss=f"{loss.item():.5f}")

    train_mse = running / max(seen, 1)
    val_mse = eval_loss(val_mon_loader)
    scheduler.step(val_mse)
    lr_now = optimizer.param_groups[0]["lr"]

    history.append(
        {
            "epoch": epoch,
            "train_mse": train_mse,
            "val_monitor_mse": val_mse,
            "lr": lr_now,
        }
    )
    print(
        f"epoch {epoch:02d}  train={train_mse:.6f}  val_mon={val_mse:.6f}  lr={lr_now:.2e}"
    )

    if val_mse + 1e-7 < best_val:
        best_val = val_mse
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f"early stop at epoch {epoch}; best epoch {best_epoch}")
            break

elapsed = time.time() - t0
print(f"training wall time: {elapsed/60:.1f} min | best_val_monitor={best_val:.6f}")

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}

unwrap(model).load_state_dict(best_state)
model.eval()

hist_df = pd.DataFrame(history)
hist_df.to_csv(OUT_DIR / "training_history.csv", index=False)

plt.figure(figsize=(7, 4))
plt.plot(hist_df["epoch"], hist_df["train_mse"], label="train")
plt.plot(hist_df["epoch"], hist_df["val_monitor_mse"], label="val monitor")
plt.xlabel("epoch")
plt.ylabel("MSE (scaled space)")
plt.title("Reconstruction loss")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "loss_curve.png", dpi=140)
plt.show()


## Clean validation error distribution

These percentiles are **candidates**, not a frozen operating threshold.
Freeze the threshold locally after simulator injections on 2023, then evaluate 2024.


In [ ]:
@torch.no_grad()
def reconstruction_errors(loader):
    model.eval()
    window_mse = []
    last_step_mse = []
    feature_mse = []  # (N, 3)
    for batch in tqdm(loader, desc="val errors"):
        batch = batch.to(DEVICE, non_blocking=True)
        recon = model(batch)
        err = (recon - batch) ** 2  # (B, T, F)
        window_mse.append(err.mean(dim=(1, 2)).cpu().numpy())
        last_step_mse.append(err[:, -1, :].mean(dim=1).cpu().numpy())
        feature_mse.append(err.mean(dim=1).cpu().numpy())
    return (
        np.concatenate(window_mse),
        np.concatenate(last_step_mse),
        np.concatenate(feature_mse, axis=0),
    )


val_window_mse, val_last_mse, val_feat_mse = reconstruction_errors(val_full_loader)

percentiles = [50, 90, 95, 97.5, 99, 99.5, 99.9]
val_percentiles = {
    "window_mse": {str(p): float(np.percentile(val_window_mse, p)) for p in percentiles},
    "last_step_mse": {str(p): float(np.percentile(val_last_mse, p)) for p in percentiles},
    "feature_mse": {
        f: {str(p): float(np.percentile(val_feat_mse[:, i], p)) for p in percentiles}
        for i, f in enumerate(FEATURES)
    },
    "n_val_windows": int(len(val_window_mse)),
    "mean_window_mse": float(val_window_mse.mean()),
    "std_window_mse": float(val_window_mse.std()),
}

print(json.dumps(val_percentiles["window_mse"], indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(val_window_mse, bins=80, color="#3b6ea5")
axes[0].set_title("Val window MSE")
axes[0].set_xlabel("MSE")
axes[1].hist(np.log10(np.clip(val_window_mse, 1e-12, None)), bins=80, color="#3b6ea5")
axes[1].set_title("Val window MSE (log10)")
axes[1].set_xlabel("log10 MSE")
for ax in axes:
    ax.set_ylabel("count")
plt.tight_layout()
plt.savefig(OUT_DIR / "val_error_hist.png", dpi=140)
plt.show()


In [ ]:
# per-station val error (for later threshold discussion — not deployed here)
per_station_err = defaultdict(list)
# map val_index alignment: reconstruction_errors concatenated in loader order = val_index order
for (sid, _start), mse in zip(val_index, val_window_mse):
    per_station_err[sid].append(float(mse))

station_err_rows = []
for sid, vals in per_station_err.items():
    a = np.asarray(vals)
    station_err_rows.append(
        {
            "station_id": sid,
            "n_val_windows": int(len(a)),
            "mean_mse": float(a.mean()),
            "p95_mse": float(np.percentile(a, 95)),
            "p99_mse": float(np.percentile(a, 99)),
        }
    )
station_err_df = pd.DataFrame(station_err_rows).sort_values("mean_mse", ascending=False)
station_err_df.to_csv(OUT_DIR / "val_error_by_station.csv", index=False)
print("highest mean val MSE stations:")
print(station_err_df.head(10).to_string(index=False))

plt.figure(figsize=(8, 4))
plt.bar(range(min(20, len(station_err_df))), station_err_df["mean_mse"].head(20))
plt.xticks(
    range(min(20, len(station_err_df))),
    station_err_df["station_id"].head(20),
    rotation=90,
)
plt.ylabel("mean val window MSE")
plt.title("Worst 20 stations on clean 2023 val")
plt.tight_layout()
plt.savefig(OUT_DIR / "val_error_worst_stations.png", dpi=140)
plt.show()


In [ ]:
def inverse_minmax(scaled_row: np.ndarray, stats: dict) -> np.ndarray:
    vmin = np.array(stats["min"], dtype=np.float64)
    vr = np.array(stats["range"], dtype=np.float64)
    return scaled_row * vr + vmin


@torch.no_grad()
def plot_reconstruction(station_id: str, start: int, title: str, fname: str):
    x = val_scaled[station_id][start : start + WINDOW_HOURS]
    xt = torch.from_numpy(np.nan_to_num(x, nan=0.0)).unsqueeze(0).to(DEVICE)
    recon = unwrap(model)(xt).squeeze(0).cpu().numpy()
    stats = scalers[station_id]
    obs = inverse_minmax(x, stats)
    hat = inverse_minmax(recon, stats)
    hours = np.arange(WINDOW_HOURS)
    fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
    for i, (ax, name) in enumerate(zip(axes, FEATURES)):
        ax.plot(hours, obs[:, i], label="observed", lw=2)
        ax.plot(hours, hat[:, i], label="reconstructed", lw=2, ls="--")
        ax.set_ylabel(name)
        ax.legend(loc="upper right")
    axes[-1].set_xlabel("hour in window")
    axes[0].set_title(title)
    plt.tight_layout()
    plt.savefig(OUT_DIR / fname, dpi=140)
    plt.show()


# one typical and one higher-error val window
if val_index:
    sid_a, st_a = val_index[len(val_index) // 2]
    plot_reconstruction(
        sid_a,
        st_a,
        f"Clean val reconstruction — {sid_a}",
        "recon_example.png",
    )
    worst_i = int(np.argmax(val_window_mse))
    sid_b, st_b = val_index[worst_i]
    plot_reconstruction(
        sid_b,
        st_b,
        f"Highest val MSE window — {sid_b}  mse={val_window_mse[worst_i]:.5f}",
        "recon_worst_val.png",
    )


## Save artifacts

Copy `/kaggle/working/artifacts/` to local `ml/artifacts/` after the run.


In [ ]:
weights_path = OUT_DIR / "lstm_autoencoder.pt"
torch.save(
    {
        "state_dict": best_state,
        "n_features": len(FEATURES),
        "hidden": HIDDEN,
        "latent": LATENT,
        "window_hours": WINDOW_HOURS,
        "features": FEATURES,
        "architecture": "lstm_ae_bottleneck",
    },
    weights_path,
)

with open(OUT_DIR / "scalers.json", "w", encoding="utf-8") as f:
    json.dump(scalers, f)

# pickle as well for convenience
import pickle

with open(OUT_DIR / "scalers.pkl", "wb") as f:
    pickle.dump(scalers, f)

with open(OUT_DIR / "val_error_percentiles.json", "w", encoding="utf-8") as f:
    json.dump(val_percentiles, f, indent=2)

metadata = {
    "seed": SEED,
    "features": FEATURES,
    "window_hours": WINDOW_HOURS,
    "train_start": TRAIN_START,
    "train_end": TRAIN_END,
    "val_start": VAL_START,
    "val_end": VAL_END,
    "test_start": TEST_START,
    "test_end": TEST_END,
    "test_used_in_this_notebook": False,
    "simulator_used": False,
    "architecture": "lstm_ae_bottleneck",
    "hidden": HIDDEN,
    "latent": LATENT,
    "train_stride": TRAIN_STRIDE,
    "val_final_stride": VAL_FINAL_STRIDE,
    "interpolate_limit_hours": INTERPOLATE_LIMIT_HOURS,
    "scaler": "station_wise_minmax_train_only",
    "clip_scaled_values": False,
    "n_stations_catalog": len(STATION_IDS),
    "n_train_stations": len(train_frames),
    "n_val_stations": len(val_frames),
    "n_train_windows": len(train_index),
    "n_val_windows": len(val_index),
    "batch_size": BATCH_SIZE,
    "epochs_ran": int(hist_df["epoch"].max()) if len(hist_df) else 0,
    "best_epoch": int(best_epoch),
    "best_val_monitor_mse": float(best_val),
    "loss": "mse",
    "optimizer": "adam",
    "lr": LR,
    "device": str(DEVICE),
    "n_gpus": int(N_GPUS),
    "raw_dir": str(RAW_DIR),
    "threshold_frozen": False,
    "note": (
        "Do not treat val percentiles as the operating threshold. "
        "Sweep on 2023 injections locally, freeze, then test 2024 with simulate_corruption."
    ),
}

with open(OUT_DIR / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Wrote:")
for p in sorted(OUT_DIR.iterdir()):
    print(" ", p.name, p.stat().st_size, "bytes")
print(json.dumps(metadata, indent=2))
